# 05 - Evaluation and Error Analysis
Compare all trained models, inspect Grad-CAM, and analyze errors.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import tensorflow as tf
import pandas as pd
from src import config
from src.data_loader import DataLoader
from src.evaluate import Evaluator
from src.gradcam import save_gradcam_samples

loader = DataLoader()
_, _, test_gen = loader.get_generators()
evalr = Evaluator()

In [ ]:
paths = {
    "custom_cnn": config.CUSTOM_MODEL_PATH,
    "resnet50_frozen": config.RESNET_FROZEN_MODEL_PATH,
    "resnet50_finetuned": config.RESNET_FINETUNE_MODEL_PATH
}
results = {}
for name, path in paths.items():
    if path.exists():
        model = tf.keras.models.load_model(str(path))
        results[name] = evalr.evaluate_model(model, test_gen, model_name=name)

comparison_df = evalr.compare_models(results)
comparison_df

In [ ]:
best_name = comparison_df.iloc[0]["model"]
best_model = tf.keras.models.load_model(str(paths[best_name]))
save_gradcam_samples(best_model, test_gen, n=5)
print("Saved Grad-CAM samples to:", config.GRADCAM_DIR)

## Final Recommendations
- Prefer the model with strongest weighted F1 and AUC-ROC.
- Keep Grad-CAM as a qualitative sanity check, not a standalone clinical explanation.
- Validate on external hospitals/domains before any real-world use.